# 🔌 Integration wiring certificate — the graduated mechanism, *through the real consolidation path* @ **D=4096** WikiText-2

**Branch:** `consolidation/role-structure` · FHRR + Modern Hopfield + emergent codebook · _Report 058_

Reports 055/056 graduated a surgical consolidation write (**heteroassociative delta-rule write + L2 cue-space decorrelator**) and proved it is a **role-selective associative memory**. Report 057 folded it into the production orchestrator `OnlineCodebookUpdater` (default-off → byte-identical). Report 058 asked the obvious next question: *does driving real data through the integrated public API reproduce the validated mechanism?* — and answered **yes, bit-identically**, locally at D=1024/2048.

**This notebook is the D=4096 graduation-scale confirmation, on GPU.** It runs a **head-to-head** on the SAME masked-encoding windows:

| | Path A (reference) | Path B (integrated) |
|---|---|---|
| write | `exp56.write_H` | `OnlineCodebookUpdater.observe(cue=)×N` → `consolidate_hetero()` |
| read | `exp56.write_read` | `recall_hetero(cue)` |
| code | the standalone code behind 055/056 | the production fold-in (Report 057) |

Both paths terminate in the **same** leaf functions with **identical** hyper-parameters (`lr=0.5, epochs=20, ridge=1e-5, β=10, max_iter=12`), so a **`torch.equal`** match on the dense `H` *and* the basin indices proves the fold-in introduced **zero computational drift**. The wiring witness is **A == B bit-identical**; the role-Selectivity-Δ is anchored to the published Report-056 D=4096 numbers (obs 1/2/3 = 0.32 / 0.73 / 0.91).

In-sample only (memorization is the target — contextual-completion, not prediction). Held-out is out of scope (real-text held-out recall ≈ chance *by design* — memory, not learner).

> ⏱️ ~5–15 min on a T4.

## 1 · GPU + fetch the code
_Clones the pushed branch (src/ substrate + exp-56). If `experiments/57_…` isn't on the branch yet, it is materialized from an embedded copy so the notebook is self-contained._

In [ ]:
EXP57_B64 = "IiIiRXhwIDU3IOKAlCBFTkQtVE8tRU5EIFdJUklORyBDSEVDSyBmb3IgdGhlIGludGVncmF0ZWQgY29uc29saWRhdGlvbiB3cml0ZS4KClJlcG9ydCAwNTcgZm9sZGVkIHRoZSB2YWxpZGF0ZWQgaGV0ZXJvYXNzb2NpYXRpdmUtd3JpdGUgKyBMMiBjdWUtc3BhY2UgZGVjb3JyZWxhdG9yCihSZXBvcnRzIDA1NSBncmFkdWF0aW9uIC8gMDU2IEctRCkgaW50byB0aGUgcHJvZHVjdGlvbiBvcmNoZXN0cmF0b3IKYGBPbmxpbmVDb2RlYm9va1VwZGF0ZXJgYCAocGhhc2UzNC9vbmxpbmVfY29kZWJvb2sucHkpLiBUaGlzIGhhcm5lc3MgdmFsaWRhdGVzIHRoZQoqd2lyaW5nKiBvZiB0aGF0IGZvbGQtaW4g4oCUIE5PVCB0aGUgbWVjaGFuaXNtLCB3aGljaCBhbHJlYWR5IGdyYWR1YXRlZC4KCkNMQVNTSUZJQ0FUSU9OOiBEUklMTC1ET1dOIC8gaW50ZWdyYXRpb24tdmFsaWRhdGlvbiwgTk9UIGEgZ3JhZHVhdGlvbiBleHBlcmltZW50LgpUaGUgY2xhaW0gdW5kZXIgdGVzdCBpcyAidGhlIGludGVncmF0ZWQgcHVibGljLUFQSSBwYXRoIHJlcHJvZHVjZXMgdGhlIHN0YW5kYWxvbmUKZXhwLTU2IGhhcm5lc3MgcmVzdWx0IChSZXBvcnRzIDA1NS8wNTYpIiwgbm90IGEgZnJlc2ggZ3JhZHVhdGlvbi4KCkRFU0lHTjogaGVhZC10by1oZWFkIGVxdWl2YWxlbmNlIG9uIGJ5dGUtaWRlbnRpY2FsIGRhdGEuCiAgRm9yIGVhY2ggKGNvcnB1cywgRCwgc2VlZCwgb2JzZXJ2ZWQpIGNlbGwsIGJ1aWxkIHRoZSBtYXNrZWQtZW5jb2Rpbmcgc3Vic3RyYXRlLCB0aGUKICB2YWx1ZSBjb2RlYm9vaywgdGhlIHBlci13aW5kb3cgUkFXIG1hc2tlZCBjdWUgS190cnVlICh0aGUga2V5KSwgdGhlIHBlci1zY2VuZQogIGZpeGVkLXBvaW50LWZyZWUgcG9zaXRpb24tZGVyYW5nZWQgY3VlIEtfZGVyICh0aGUgcm9sZS1zaHVmZmxlIGFybSksIGFuZCB0aGUKICB2YWx1ZS1jb2RlYm9vay1sb2NhbCB0YXJnZXRzIHRndCDigJQgRVhBQ1RMWSBhcyBleHBlcmltZW50cy81NiBidWlsZHMgdGhlbSAod2UgcmV1c2UKICBleHAtNTYncyBvd24gVG9waWNDb3JwdXMgLyBDb3JwdXNXaW5kb3dzIC8gZW5jb2RlX2N1ZSAvIHdyaXRlX0ggLyB3cml0ZV9yZWFkIC8KICBidWlsZF9wb3NpdGlvbl92ZWN0b3JzKS4gVGhlbiB3ZSBydW4gdGhlIFNBTUUgZGF0YSB0aHJvdWdoIHR3byBwYXRoczoKCiAgICBQYXRoIEEgKHN0YW5kYWxvbmUgcmVmZXJlbmNlKTogZXhwNTYud3JpdGVfSCArIGV4cDU2LndyaXRlX3JlYWQgICh0aGUgY29kZSB0aGF0CiAgICAgICAgICAgIHByb2R1Y2VkIFJlcG9ydHMgMDU1LzA1NikuCiAgICBQYXRoIEIgKGludGVncmF0ZWQpOiBPbmxpbmVDb2RlYm9va1VwZGF0ZXIub2JzZXJ2ZShjdWU9KSB4IE4gLT4gY29uc29saWRhdGVfaGV0ZXJvKCkKICAgICAgICAgICAgLT4gcmVjYWxsX2hldGVybyhjdWUpICAodGhlIHByb2R1Y3Rpb24gZm9sZC1pbiwgUmVwb3J0IDA1NykuCgogIEJvdGggcGF0aHMgY2FsbCB0aGUgU0FNRSBwaGFzZTQuaGV0ZXJvX3dyaXRlLmhldGVyb2Fzc29jaWF0aXZlX3dyaXRlICsKICByZWNhbGxfdG9wX2luZGV4ICsgQ3VlRGVjb3JyZWxhdG9yIHdpdGggSURFTlRJQ0FMIGRlZmF1bHRzIChscj0wLjUsIGVwb2Nocz0yMCwKICByaWRnZT0xZS01LCBiZXRhPTEwLCBtYXhfaXRlcj0xMiksIHNvIG9uIGlkZW50aWNhbCBkYXRhIHRoZXkgbXVzdCBwcm9kdWNlCiAgQklULUlERU5USUNBTCBiYXNpbiBpbmRpY2VzLiBXZSBhc3NlcnQgdG9yY2guZXF1YWwgb24gdGhlIGJhc2luIGluZGljZXMgYW5kIEgsIGFuZAogIHJlY29tcHV0ZSB0aGUgdHdvLWZsb29yIHJvbGUtU2VsZWN0aXZpdHktzpQgZnJvbSB0aGUgaW50ZWdyYXRlZCBwYXRoJ3Mgb3duIG91dHB1dC4KCiAgS0VZIE1BUFBJTkc6IHRoZSBpbnRlZ3JhdGVkIHBhdGggd3JpdGVzIHRhcmdldHMgdmlhIGBgc2VsZi5jb2RlYm9va1t2aWR4XWBgIGFuZCByZWFkcwogIGBgdG9wX2luZGV4YGAgb3ZlciBgYHNlbGYuY29kZWJvb2tgYCwgd2hlcmVhcyBleHAtNTYgdXNlcyBgYHZhbHVlX2NiYGAuIFNvIFBhdGggQiBpcwogIGNvbnN0cnVjdGVkIHdpdGggYGBjb2RlYm9vaz12YWx1ZV9jYmBgIGFuZCBgYHRhcmdldF9pZGBgID0gdGhlIHZhbHVlLWNvZGVib29rLWxvY2FsCiAgaW5kZXguIFRoaXMgaXMgdGhlIGxvYWQtYmVhcmluZyBlcXVpdmFsZW5jZSBtYXBwaW5nLgoKSW4tc2FtcGxlIG9ubHkgKHRyPT10ZT09cmFuZ2UoTikpOiB0aGUgdGFyZ2V0IGNhcGFiaWxpdHkgaXMgbWVtb3JpemF0aW9uLXJlY2FsbAooQ0xBVURFLm1kIGNvbnRleHR1YWwtY29tcGxldGlvbiwgTk9UIHByZWRpY3Rpb24pOyBoZWxkLW91dCBpcyBvdXQgb2Ygc2NvcGUgZm9yIGEKd2lyaW5nIGNoZWNrIChyZWFsLXRleHQgaGVsZC1vdXQgcmVjYWxsIGlzIH5jaGFuY2UgYnkgZGVzaWduIOKAlCBtZW1vcnkgbm90IGxlYXJuZXIpLgoKUnVuOgogIFBZVEhPTlBBVEg9c3JjIC52ZW52L2Jpbi9weXRob24gZXhwZXJpbWVudHMvNTdfZTJlX2ludGVncmF0aW9uX3dpcmluZ19jaGVjay5weSBcCiAgICAgIC0tZGV2aWNlIGNwdSAtLW91dCByZXBvcnRzLzA1OF9lMmVfaW50ZWdyYXRpb25fd2lyaW5nL3dpcmluZ19yZXN1bHRzLmpzb24KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaW1wb3J0bGliLnV0aWwKaW1wb3J0IGpzb24KaW1wb3J0IHBhdGhsaWIKaW1wb3J0IHN5cwoKaW1wb3J0IHRvcmNoCgpSRVBPID0gcGF0aGxpYi5QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJFUE8gLyAic3JjIikpCgojIGV4cC01NidzIG1vZHVsZSBuYW1lIHN0YXJ0cyB3aXRoIGEgZGlnaXQgLT4gbG9hZCBieSBwYXRoLgpfc3BlYyA9IGltcG9ydGxpYi51dGlsLnNwZWNfZnJvbV9maWxlX2xvY2F0aW9uKAogICAgImV4cDU2X3BhbmVsIiwgUkVQTyAvICJleHBlcmltZW50cyIgLyAiNTZfZ2Rfc2VsZWN0aXZpdHlfcGFuZWwucHkiKQpleHA1NiA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoX3NwZWMpCl9zcGVjLmxvYWRlci5leGVjX21vZHVsZShleHA1NikKCmZyb20gZW5lcmd5X21lbW9yeS5zdWJzdHJhdGUudG9yY2hfZmhyciBpbXBvcnQgVG9yY2hGSFJSCmZyb20gZW5lcmd5X21lbW9yeS5waGFzZTMuYmFzaW5fcmVhZG91dCBpbXBvcnQgdG9wX2luZGV4X2hpdHMsIHNlbGVjdGl2aXR5X2RlbHRhCmZyb20gZW5lcmd5X21lbW9yeS5waGFzZTM0Lm9ubGluZV9jb2RlYm9vayBpbXBvcnQgT25saW5lQ29kZWJvb2tVcGRhdGVyCgojIFJldXNlIGV4cC01NidzIGV4YWN0IGJ1aWxkaW5nIGJsb2NrcyAoZ3VhcmFudGVlcyBieXRlLWlkZW50aWNhbCBkYXRhICsgcmVmZXJlbmNlIHBhdGgpLgpUb3BpY0NvcnB1cyA9IGV4cDU2LlRvcGljQ29ycHVzCkNvcnB1c1dpbmRvd3MgPSBleHA1Ni5Db3JwdXNXaW5kb3dzCmVuY29kZV9jdWUgPSBleHA1Ni5lbmNvZGVfY3VlCmJ1aWxkX3Bvc2l0aW9uX3ZlY3RvcnMgPSBleHA1Ni5idWlsZF9wb3NpdGlvbl92ZWN0b3JzCndyaXRlX0ggPSBleHA1Ni53cml0ZV9ICndyaXRlX3JlYWQgPSBleHA1Ni53cml0ZV9yZWFkCmJhdGNoZWRfaG9wZmllbGRfdG9waW5kZXggPSBleHA1Ni5iYXRjaGVkX2hvcGZpZWxkX3RvcGluZGV4CgoKZGVmIGJ1aWxkX2RhdGEoY29ycHVzLCBELCBzZWVkLCBvYnNlcnZlZCwgVywgZGV2LCAqLCBMLCBWYywgTiwgZXBzLCBtYXhfdm9jYWIpOgogICAgIiIiUmVwbGljYXRlIGV4YWN0bHkgdGhlIChzZWVkLCBvYnNlcnZlZCkgZGF0YSBjb25zdHJ1Y3Rpb24gb2YgZXhwNTYucnVuX2NlbGwsCiAgICBpbi1zYW1wbGUgKHRyPT10ZT09cmFuZ2UoTikpLiBSZXR1cm5zIHN1YiwgdmFsdWVfY2IsIHRndCwgY2hhbmNlLCBLX3RydWUsIEtfZGVyLiIiIgogICAgaWYgY29ycHVzID09ICJzeW50aGV0aWMiOgogICAgICAgIHNyYyA9IFRvcGljQ29ycHVzKHNlZWQ9c2VlZCwgVz1XLCBMPUwsIFZjPVZjLCBOPU4sIGVwcz1lcHMsIHBvc2RlcD1UcnVlKQogICAgZWxzZToKICAgICAgICBzcmMgPSBDb3JwdXNXaW5kb3dzKHNlZWQ9c2VlZCwgVz1XLCBOPU4sIGNvcnB1c19zb3VyY2U9Y29ycHVzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgd2lraXRleHRfbmFtZT0id2lraXRleHQtMi1yYXctdjEiLCBtYXhfdm9jYWI9bWF4X3ZvY2FiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVwb19yb290PVJFUE8pCiAgICBvcCA9IHNyYy5vYnNlcnZlZF9wb3NpdGlvbnMob2JzZXJ2ZWQpCiAgICBtcG9zLCBtYXNrX2lkLCBXID0gc3JjLm1wb3MsIHNyYy5tYXNrX2lkLCBzcmMuVwogICAgc3ViID0gVG9yY2hGSFJSKGRpbT1ELCBzZWVkPXNlZWQsIGRldmljZT1kZXYpCiAgICBjb2RlYm9vaywgdmFsdWVfY2IsIHRndCwgY2hhbmNlID0gc3JjLmNvZGVib29rX3RhcmdldHMoc3ViKQogICAgdGd0ID0gdGd0LnRvKGRldikKICAgIHBvc2l0aW9ucyA9IGJ1aWxkX3Bvc2l0aW9uX3ZlY3RvcnMoc3ViLCBXKQoKICAgIGRlZiBjdWVfc2V0KG1vZGUpOgogICAgICAgIHJldHVybiB0b3JjaC5zdGFjayhbZW5jb2RlX2N1ZShzdWIsIHBvc2l0aW9ucywgY29kZWJvb2ssIHcsIG9wLCBtcG9zLCBXLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXNrX2lkLCBtb2RlPW1vZGUpIGZvciB3IGluIHNyYy53aW5kb3dzXSkKCiAgICBLX3RydWUgPSBjdWVfc2V0KCJ0cnVlIikKCiAgICAjIFBlci1zY2VuZSBmaXhlZC1wb2ludC1mcmVlIHBvc2l0aW9uIGRlcmFuZ2VtZW50ID0gdGhlIHJvbGUtc2h1ZmZsZSAoQ29udHJvbCAzKQogICAgIyBhcm0uIFZlcmJhdGltIGNvcHkgb2YgZXhwNTYucnVuX2NlbGwuZGVyYW5nZWRfY3VlX3NldCAoc2FtZSBzZWVkIGZvcm11bGEpLgogICAgZGVmIGRlcmFuZ2VkX2N1ZV9zZXQoKToKICAgICAgICBrZXlzID0gW10KICAgICAgICBmb3Igc2ksIHcgaW4gZW51bWVyYXRlKHNyYy53aW5kb3dzKToKICAgICAgICAgICAgZyA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKHNlZWQgKiAxMDAwMDMgKyBzaSAqIDEzMSArIDE3KQogICAgICAgICAgICBycCA9IGxpc3QocmFuZ2UoVykpCiAgICAgICAgICAgIGlmIGxlbihvcCkgPj0gMjoKICAgICAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICAgICAgcHAgPSB0b3JjaC5yYW5kcGVybShsZW4ob3ApLCBnZW5lcmF0b3I9ZykudG9saXN0KCkKICAgICAgICAgICAgICAgICAgICBpZiBhbGwoaSAhPSBwIGZvciBpLCBwIGluIGVudW1lcmF0ZShwcCkpOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgZm9yIGksIHAgaW4gZW51bWVyYXRlKG9wKToKICAgICAgICAgICAgICAgICAgICBycFtwXSA9IG9wW3BwW2ldXQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgY2FuZHMgPSBbcCBmb3IgcCBpbiByYW5nZShXKSBpZiBwICE9IG9wWzBdXQogICAgICAgICAgICAgICAgcnBbb3BbMF1dID0gY2FuZHNbaW50KHRvcmNoLnJhbmRpbnQoMCwgbGVuKGNhbmRzKSwgKDEsKSwgZ2VuZXJhdG9yPWcpKV0KICAgICAgICAgICAga2V5cy5hcHBlbmQoZW5jb2RlX2N1ZShzdWIsIHBvc2l0aW9ucywgY29kZWJvb2ssIHcsIG9wLCBtcG9zLCBXLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1hc2tfaWQsIG1vZGU9InRydWUiLCBwZXJtPXJwKSkKICAgICAgICByZXR1cm4gdG9yY2guc3RhY2soa2V5cykKCiAgICBLX2RlciA9IGRlcmFuZ2VkX2N1ZV9zZXQoKQogICAgcmV0dXJuIHN1YiwgdmFsdWVfY2IsIHRndCwgZmxvYXQoY2hhbmNlKSwgS190cnVlLCBLX2RlcgoKCmRlZiBydW5fcGF0aHMoc3ViLCB2YWx1ZV9jYiwgdGd0LCBLX3RydWUsIEtfZGVyLCAqLCBELCBsciwgZXBvY2hzLCBiZXRhLCBtaSk6CiAgICAiIiJSdW4gdGhlIFNBTUUgZGF0YSB0aHJvdWdoIFBhdGggQSAoc3RhbmRhbG9uZSkgYW5kIFBhdGggQiAoaW50ZWdyYXRlZCkuIiIiCiAgICBOID0gS190cnVlLnNoYXBlWzBdCgogICAgIyAtLS0tIFBhdGggQTogdGhlIHN0YW5kYWxvbmUgZXhwLTU2IHJlZmVyZW5jZSAodGhlIGNvZGUgYmVoaW5kIFJlcG9ydHMgMDU1LzA1NikgLS0tLQogICAgSF9hLCBkZWNfYSA9IHdyaXRlX0goS190cnVlLCB0Z3QsIHZhbHVlX2NiLCBkZWNvcnI9ImwyIiwgZGltPUQsIGxyPWxyLCBlcG9jaHM9ZXBvY2hzKQogICAgdGlfdHJ1ZV9hID0gd3JpdGVfcmVhZChzdWIsIEhfYSwgZGVjX2EsIEtfdHJ1ZSwgdmFsdWVfY2IsIGJldGEsIG1pKVswXQogICAgdGlfZGVyX2EgPSB3cml0ZV9yZWFkKHN1YiwgSF9hLCBkZWNfYSwgS19kZXIsIHZhbHVlX2NiLCBiZXRhLCBtaSlbMF0KCiAgICAjIC0tLS0gUGF0aCBCOiB0aGUgaW50ZWdyYXRlZCBwcm9kdWN0aW9uIGZvbGQtaW4gKFJlcG9ydCAwNTcgcHVibGljIEFQSSkgLS0tLQogICAgdXBkID0gT25saW5lQ29kZWJvb2tVcGRhdGVyKAogICAgICAgIHN1YiwgdmFsdWVfY2IsCiAgICAgICAgaGV0ZXJvX3dyaXRlX2VuYWJsZWQ9VHJ1ZSwKICAgICAgICBkZWNvcnJlbGF0b3JfZW5hYmxlZD1UcnVlLAogICAgICAgIGhldGVyb19scj1sciwKICAgICAgICBoZXRlcm9fZXBvY2hzPWVwb2NocywKICAgICAgICBoZXRlcm9fY29udHJhc3RpdmU9RmFsc2UsCiAgICAgICAgZGVjb3JyZWxhdG9yX3JpZGdlPTFlLTUsCiAgICApCiAgICBmb3IgaSBpbiByYW5nZShOKToKICAgICAgICB0ID0gaW50KHRndFtpXSkKICAgICAgICAjIHNsb3RfcXVlcnkvcHJlZGljdGVkX2lkIGZlZWQgb25seSB0aGUgbGVnYWN5IHB1bGwvcHVzaCBnYXRlOyBwYXNzaW5nIHRoZQogICAgICAgICMgdGFyZ2V0IGF0b20ga2VlcHMgdGhhdCBwYXRoIGluZXJ0IChxdWFsaXR5IGhpZ2gpLiBUaGUgaGV0ZXJvIHdyaXRlIHVzZXMgY3VlLgogICAgICAgIHVwZC5vYnNlcnZlKHRhcmdldF9pZD10LCBzbG90X3F1ZXJ5PXZhbHVlX2NiW3RdLCBwcmVkaWN0ZWRfaWQ9dCwgY3VlPUtfdHJ1ZVtpXSkKICAgIG1ldGEgPSB1cGQuY29uc29saWRhdGVfaGV0ZXJvKCkKICAgIHRpX3RydWVfYiA9IHVwZC5yZWNhbGxfaGV0ZXJvKEtfdHJ1ZSwgYmV0YT1iZXRhLCBtYXhfaXRlcj1taSlbMF0KICAgIHRpX2Rlcl9iID0gdXBkLnJlY2FsbF9oZXRlcm8oS19kZXIsIGJldGE9YmV0YSwgbWF4X2l0ZXI9bWkpWzBdCgogICAgIyAtLS0tIEVxdWl2YWxlbmNlIGRpYWdub3N0aWNzIC0tLS0KICAgIEhfZXF1YWwgPSBib29sKHRvcmNoLmVxdWFsKEhfYSwgdXBkLmhldGVyb19IKSkKICAgIEhfYWxsY2xvc2UgPSBib29sKHRvcmNoLmFsbGNsb3NlKEhfYSwgdXBkLmhldGVyb19ILCBhdG9sPTFlLTYsIHJ0b2w9MWUtNSkpCiAgICB0aXhfdHJ1ZV9lcXVhbCA9IGJvb2wodG9yY2guZXF1YWwodGlfdHJ1ZV9hLCB0aV90cnVlX2IpKQogICAgdGl4X2Rlcl9lcXVhbCA9IGJvb2wodG9yY2guZXF1YWwodGlfZGVyX2EsIHRpX2Rlcl9iKSkKCiAgICAjIC0tLS0gSW50ZWdyYXRlZC1wYXRoIHJlYWRvdXQtbGVhayBjb250cm9sOiByZWFkIHRoZSBpbnRlZ3JhdGVkIEggKyBpbnRlZ3JhdGVkCiAgICAjIGRlY29ycmVsYXRvciBhZ2FpbnN0IGEgRlJFU0ggcmFuZG9tIGNvZGVib29rIC0+IG11c3QgY29sbGFwc2UgdG8gY2hhbmNlLiAtLS0tCiAgICByYW5kX2NiID0gc3ViLnJhbmRvbV92ZWN0b3JzKHZhbHVlX2NiLnNoYXBlWzBdKQogICAgcmVjYWxsZWQgPSAodXBkLmhldGVyb19kZWNvcnJlbGF0b3IuYXBwbHkoS190cnVlKSBAIHVwZC5oZXRlcm9fSC5UKSAvIEQKICAgIHRpX3JhbmQgPSBiYXRjaGVkX2hvcGZpZWxkX3RvcGluZGV4KHN1YiwgcmFuZF9jYiwgcmVjYWxsZWQsIGJldGE9YmV0YSwgbWF4X2l0ZXI9bWkpWzBdCiAgICByYW5kX2hpdHMgPSB0b3BfaW5kZXhfaGl0cyh0aV9yYW5kLCB0Z3QpCgogICAgcmV0dXJuIHsKICAgICAgICAibWV0YSI6IG1ldGEsCiAgICAgICAgIkhfZXF1YWwiOiBIX2VxdWFsLAogICAgICAgICJIX2FsbGNsb3NlIjogSF9hbGxjbG9zZSwKICAgICAgICAidGl4X3RydWVfZXF1YWwiOiB0aXhfdHJ1ZV9lcXVhbCwKICAgICAgICAidGl4X2Rlcl9lcXVhbCI6IHRpeF9kZXJfZXF1YWwsCiAgICAgICAgIyByYXcgaGl0cyBmb3IgcG9vbGluZyAoaW50ZWdyYXRlZCBQYXRoIEIpCiAgICAgICAgIkJfdHJ1ZV9oaXRzIjogdG9wX2luZGV4X2hpdHModGlfdHJ1ZV9iLCB0Z3QpLAogICAgICAgICJCX2Rlcl9oaXRzIjogdG9wX2luZGV4X2hpdHModGlfZGVyX2IsIHRndCksCiAgICAgICAgIyByYXcgaGl0cyBQYXRoIEEgKGZvciB0aGUgcGVyLWNlbGwgcG9pbnQgYW5jaG9yKQogICAgICAgICJBX3RydWVfaGl0cyI6IHRvcF9pbmRleF9oaXRzKHRpX3RydWVfYSwgdGd0KSwKICAgICAgICAiQV9kZXJfaGl0cyI6IHRvcF9pbmRleF9oaXRzKHRpX2Rlcl9hLCB0Z3QpLAogICAgICAgICJyYW5kX2hpdHMiOiByYW5kX2hpdHMsCiAgICAgICAgIm4iOiBOLAogICAgfQoKCiMgQ2Fub25pY2FsIGluLXNhbXBsZSBjZWxscyBhbmNob3JlZCB0byBSZXBvcnRzIDA1NS8wNTYgKHRoZSByZXBvcnQgc2VlZC9jb25maWcpLgpQUkVTRVRTID0gewogICAgInJlcG9fc2FtcGxlIjogZGljdChEPTEwMjQsIE49NTAwLCBXPTYsIG1heF92b2NhYj01MTIsIEw9OCwgVmM9MTYsIGVwcz0wLjI1LAogICAgICAgICAgICAgICAgICAgICAgICBzZWVkcz1bMCwgMSwgMl0sIG9ic2VydmVkPVsxLCAyLCAzXSksCiAgICAic3ludGhldGljIjogICBkaWN0KEQ9MjA0OCwgTj01MDAsIFc9NiwgbWF4X3ZvY2FiPTUxMiwgTD04LCBWYz0xNiwgZXBzPTAuMjUsCiAgICAgICAgICAgICAgICAgICAgICAgIHNlZWRzPVswLCAxLCAyLCAzLCA0XSwgb2JzZXJ2ZWQ9WzEsIDIsIDNdKSwgICMgNSBzZWVkcyA9IFJlcG9ydC0wNTYgY29uZmlnCiAgICAjIEdyYWR1YXRpb24tc2NhbGUgY29udmluY2VyIChSZXBvcnRzIDA1NS8wNTYgV2lraVRleHQtMiBEPTQwOTYgY29uZmlnKS4gQ1VEQS1vbmx5CiAgICAjIGluIHByYWN0aWNlIChjb21wbGV4IGVpZ2ggKyBkZW5zZSBIIEAgRD00MDk2KTsgc2VlIG5vdGVib29rcy8wNThfKi5pcHluYiBmb3IgQ29sYWIuCiAgICAid2lraXRleHQiOiAgICBkaWN0KEQ9NDA5NiwgTj0xMDAwLCBXPTYsIG1heF92b2NhYj0yMDAwLCBMPTgsIFZjPTE2LCBlcHM9MC4yNSwKICAgICAgICAgICAgICAgICAgICAgICAgc2VlZHM9WzAsIDEsIDJdLCBvYnNlcnZlZD1bMSwgMiwgM10pLAp9CgojIFJlcG9ydC0wNTUvMDU2IGluLXNhbXBsZSBwb29sZWQgcm9sZS1TZWxlY3Rpdml0eS3OlCB0YXJnZXRzIChmb3IgdGhlIGFuY2hvciBwcmludCkuClJFUE9SVF9UQVJHRVRTID0gewogICAgKCJyZXBvX3NhbXBsZSIsIDEpOiAwLjM1MiwgKCJyZXBvX3NhbXBsZSIsIDIpOiAwLjc3NSwgKCJyZXBvX3NhbXBsZSIsIDMpOiAwLjkzMywKICAgICgic3ludGhldGljIiwgMik6IDAuNTkyLAogICAgIyBSZXBvcnQtMDU2IEctRCBEPTQwOTYgV2lraVRleHQgcm9sZS1TZWxlY3Rpdml0eS3OlCAodHJ1ZS1wb3Mg4oiSIGRlcmFuZ2VkKS4KICAgICgid2lraXRleHQiLCAxKTogMC4zMjMsICgid2lraXRleHQiLCAyKTogMC43MzAsICgid2lraXRleHQiLCAzKTogMC45MDcsCn0KCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD0iY3B1IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb3Jwb3JhIiwgbmFyZ3M9IisiLAogICAgICAgICAgICAgICAgICAgIGNob2ljZXM9WyJyZXBvX3NhbXBsZSIsICJzeW50aGV0aWMiLCAid2lraXRleHQiXSwKICAgICAgICAgICAgICAgICAgICBkZWZhdWx0PVsicmVwb19zYW1wbGUiLCAic3ludGhldGljIl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD0yMCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iZXRhIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xMC4wKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1pIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTIpCiAgICAjIE9wdGlvbmFsIHBlci1ydW4gb3ZlcnJpZGVzIG9mIHRoZSBjaG9zZW4gcHJlc2V0KHMpICgwL05vbmUgLT4ga2VlcCBwcmVzZXQpLgogICAgYXAuYWRkX2FyZ3VtZW50KCItLW92ZXJyaWRlLUQiLCB0eXBlPWludCwgZGVmYXVsdD0wLCBkZXN0PSJvdmVycmlkZV9EIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdmVycmlkZS1OIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCwgZGVzdD0ib3ZlcnJpZGVfTiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3ZlcnJpZGUtbWF4LXZvY2FiIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCwgZGVzdD0ib3ZlcnJpZGVfbWF4X3ZvY2FiIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkcyIsIHR5cGU9aW50LCBkZWZhdWx0PTAsIGhlbHA9InNlZWQgY291bnQgKDAgLT4gcHJlc2V0KSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb2JzZXJ2ZWQiLCB0eXBlPWludCwgbmFyZ3M9IisiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0iIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICByZXN1bHRzID0geyJjb25maWciOiB2YXJzKGFyZ3MpLCAiY2VsbHMiOiBbXSwgInBvb2xlZCI6IHt9fQogICAgYWxsX2VxdWFsID0gVHJ1ZQoKICAgIGZvciBjb3JwdXMgaW4gYXJncy5jb3Jwb3JhOgogICAgICAgIHAgPSBkaWN0KFBSRVNFVFNbY29ycHVzXSkgICMgY29weSBzbyBvdmVycmlkZXMgZG9uJ3QgbXV0YXRlIHRoZSBwcmVzZXQKICAgICAgICBpZiBhcmdzLm92ZXJyaWRlX0Q6CiAgICAgICAgICAgIHBbIkQiXSA9IGFyZ3Mub3ZlcnJpZGVfRAogICAgICAgIGlmIGFyZ3Mub3ZlcnJpZGVfTjoKICAgICAgICAgICAgcFsiTiJdID0gYXJncy5vdmVycmlkZV9OCiAgICAgICAgaWYgYXJncy5vdmVycmlkZV9tYXhfdm9jYWI6CiAgICAgICAgICAgIHBbIm1heF92b2NhYiJdID0gYXJncy5vdmVycmlkZV9tYXhfdm9jYWIKICAgICAgICBpZiBhcmdzLnNlZWRzOgogICAgICAgICAgICBwWyJzZWVkcyJdID0gbGlzdChyYW5nZShhcmdzLnNlZWRzKSkKICAgICAgICBpZiBhcmdzLm9ic2VydmVkOgogICAgICAgICAgICBwWyJvYnNlcnZlZCJdID0gYXJncy5vYnNlcnZlZAogICAgICAgICMgcG9vbGVkIGFjY3VtdWxhdG9ycyBwZXIgb2JzZXJ2ZWQgbGV2ZWwKICAgICAgICBwb29sID0ge29iczogZGljdChBX3RydWU9MCwgQV9kZXI9MCwgQl90cnVlPTAsIEJfZGVyPTAsIHJhbmQ9MCwgbj0wKQogICAgICAgICAgICAgICAgZm9yIG9icyBpbiBwWyJvYnNlcnZlZCJdfQogICAgICAgIGZvciBzZWVkIGluIHBbInNlZWRzIl06CiAgICAgICAgICAgIGZvciBvYnMgaW4gcFsib2JzZXJ2ZWQiXToKICAgICAgICAgICAgICAgIHN1YiwgdmFsdWVfY2IsIHRndCwgY2hhbmNlLCBLX3RydWUsIEtfZGVyID0gYnVpbGRfZGF0YSgKICAgICAgICAgICAgICAgICAgICBjb3JwdXMsIHBbIkQiXSwgc2VlZCwgb2JzLCBwWyJXIl0sIGFyZ3MuZGV2aWNlLAogICAgICAgICAgICAgICAgICAgIEw9cFsiTCJdLCBWYz1wWyJWYyJdLCBOPXBbIk4iXSwgZXBzPXBbImVwcyJdLCBtYXhfdm9jYWI9cFsibWF4X3ZvY2FiIl0pCiAgICAgICAgICAgICAgICByID0gcnVuX3BhdGhzKHN1YiwgdmFsdWVfY2IsIHRndCwgS190cnVlLCBLX2RlciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgRD1wWyJEIl0sIGxyPWFyZ3MubHIsIGVwb2Nocz1hcmdzLmVwb2NocywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYmV0YT1hcmdzLmJldGEsIG1pPWFyZ3MubWkpCiAgICAgICAgICAgICAgICBjZWxsX2VxdWFsID0gclsidGl4X3RydWVfZXF1YWwiXSBhbmQgclsidGl4X2Rlcl9lcXVhbCJdCiAgICAgICAgICAgICAgICBhbGxfZXF1YWwgPSBhbGxfZXF1YWwgYW5kIGNlbGxfZXF1YWwKICAgICAgICAgICAgICAgIGNlbGwgPSB7ImNvcnB1cyI6IGNvcnB1cywgIkQiOiBwWyJEIl0sICJzZWVkIjogc2VlZCwgIm9ic2VydmVkIjogb2JzLAogICAgICAgICAgICAgICAgICAgICAgICAiY2hhbmNlIjogY2hhbmNlLCAibiI6IHJbIm4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgInZvY2FiIjogaW50KHZhbHVlX2NiLnNoYXBlWzBdKSwKICAgICAgICAgICAgICAgICAgICAgICAgIkhfZXF1YWwiOiByWyJIX2VxdWFsIl0sICJIX2FsbGNsb3NlIjogclsiSF9hbGxjbG9zZSJdLAogICAgICAgICAgICAgICAgICAgICAgICAidGl4X3RydWVfZXF1YWwiOiByWyJ0aXhfdHJ1ZV9lcXVhbCJdLAogICAgICAgICAgICAgICAgICAgICAgICAidGl4X2Rlcl9lcXVhbCI6IHJbInRpeF9kZXJfZXF1YWwiXSwKICAgICAgICAgICAgICAgICAgICAgICAgIkJfdHJ1ZV9yYXRlIjogclsiQl90cnVlX2hpdHMiXSAvIHJbIm4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgIkJfZGVyX3JhdGUiOiByWyJCX2Rlcl9oaXRzIl0gLyByWyJuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICJBX3RydWVfcmF0ZSI6IHJbIkFfdHJ1ZV9oaXRzIl0gLyByWyJuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICJyYW5kX3JhdGUiOiByWyJyYW5kX2hpdHMiXSAvIHJbIm4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgImNvbnNvbGlkYXRlX21ldGEiOiByWyJtZXRhIl19CiAgICAgICAgICAgICAgICByZXN1bHRzWyJjZWxscyJdLmFwcGVuZChjZWxsKQogICAgICAgICAgICAgICAgcGsgPSBwb29sW29ic10KICAgICAgICAgICAgICAgIHBrWyJBX3RydWUiXSArPSByWyJBX3RydWVfaGl0cyJdOyBwa1siQV9kZXIiXSArPSByWyJBX2Rlcl9oaXRzIl0KICAgICAgICAgICAgICAgIHBrWyJCX3RydWUiXSArPSByWyJCX3RydWVfaGl0cyJdOyBwa1siQl9kZXIiXSArPSByWyJCX2Rlcl9oaXRzIl0KICAgICAgICAgICAgICAgIHBrWyJyYW5kIl0gKz0gclsicmFuZF9oaXRzIl07IHBrWyJuIl0gKz0gclsibiJdCiAgICAgICAgICAgICAgICBwcmludChmIlt7Y29ycHVzfSBEe3BbJ0QnXX0gc2VlZHtzZWVkfSBvYnN7b2JzfV0gIgogICAgICAgICAgICAgICAgICAgICAgZiJ0aXhfZXEodHJ1ZS9kZXIpPXtyWyd0aXhfdHJ1ZV9lcXVhbCddfS97clsndGl4X2Rlcl9lcXVhbCddfSAiCiAgICAgICAgICAgICAgICAgICAgICBmIkhfZXE9e3JbJ0hfZXF1YWwnXX0gQl90cnVlPXtjZWxsWydCX3RydWVfcmF0ZSddOi4zZn0gIgogICAgICAgICAgICAgICAgICAgICAgZiJCX2Rlcj17Y2VsbFsnQl9kZXJfcmF0ZSddOi4zZn0gcmFuZD17Y2VsbFsncmFuZF9yYXRlJ106LjRmfSAiCiAgICAgICAgICAgICAgICAgICAgICBmImNoYW5jZT17Y2hhbmNlOi40Zn0iLCBmaWxlPXN5cy5zdGRlcnIsIGZsdXNoPVRydWUpCgogICAgICAgICMgcG9vbGVkIHR3by1mbG9vciByZWFkIHBlciBvYnNlcnZlZCAoUGF0aCBCID0gaW50ZWdyYXRlZCkKICAgICAgICBmb3Igb2JzLCBwayBpbiBwb29sLml0ZW1zKCk6CiAgICAgICAgICAgIGNoYW5jZSA9IG5leHQoY1siY2hhbmNlIl0gZm9yIGMgaW4gcmVzdWx0c1siY2VsbHMiXQogICAgICAgICAgICAgICAgICAgICAgICAgIGlmIGNbImNvcnB1cyJdID09IGNvcnB1cyBhbmQgY1sib2JzZXJ2ZWQiXSA9PSBvYnMpCiAgICAgICAgICAgIHNkX2IgPSBzZWxlY3Rpdml0eV9kZWx0YSh0cnVlX2hpdHM9cGtbIkJfdHJ1ZSJdLCB0cnVlX249cGtbIm4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNodWZmbGVkX2hpdHM9cGtbIkJfZGVyIl0sIHNodWZmbGVkX249cGtbIm4iXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNoYW5jZT1jaGFuY2UpLmFzX2RpY3QoKQogICAgICAgICAgICBzZF9hID0gc2VsZWN0aXZpdHlfZGVsdGEodHJ1ZV9oaXRzPXBrWyJBX3RydWUiXSwgdHJ1ZV9uPXBrWyJuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzaHVmZmxlZF9oaXRzPXBrWyJBX2RlciJdLCBzaHVmZmxlZF9uPXBrWyJuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjaGFuY2U9Y2hhbmNlKS5hc19kaWN0KCkKICAgICAgICAgICAgdGd0X3JlcG9ydCA9IFJFUE9SVF9UQVJHRVRTLmdldCgoY29ycHVzLCBvYnMpKQogICAgICAgICAgICByZXN1bHRzWyJwb29sZWQiXVtmIntjb3JwdXN9X29ic3tvYnN9Il0gPSB7CiAgICAgICAgICAgICAgICAiaW50ZWdyYXRlZF9CIjogc2RfYiwgInN0YW5kYWxvbmVfQSI6IHNkX2EsCiAgICAgICAgICAgICAgICAicmFuZF9yYXRlIjogcGtbInJhbmQiXSAvIHBrWyJuIl0sICJuX3Bvb2xlZCI6IHBrWyJuIl0sCiAgICAgICAgICAgICAgICAicmVwb3J0X3JvbGVfZGVsdGEiOiB0Z3RfcmVwb3J0LAogICAgICAgICAgICAgICAgIkFfbWludXNfcmVwb3J0IjogKHNkX2FbImRlbHRhIl0gLSB0Z3RfcmVwb3J0KSBpZiB0Z3RfcmVwb3J0IGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgICJCX2VxdWFsc19BIjogYWJzKHNkX2JbImRlbHRhIl0gLSBzZF9hWyJkZWx0YSJdKSA8IDFlLTksCiAgICAgICAgICAgIH0KCiAgICByZXN1bHRzWyJhbGxfYmFzaW5faW5kaWNlc19iaXRfaWRlbnRpY2FsIl0gPSBhbGxfZXF1YWwKICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0c1sicG9vbGVkIl0sIGluZGVudD0yKSkKICAgIHByaW50KGYiXG5BTEwgYmFzaW4gaW5kaWNlcyBiaXQtaWRlbnRpY2FsIChBPT1CKSBhY3Jvc3MgZXZlcnkgY2VsbDoge2FsbF9lcXVhbH0iKQogICAgaWYgYXJncy5vdXQ6CiAgICAgICAgb3V0cCA9IHBhdGhsaWIuUGF0aChhcmdzLm91dCkKICAgICAgICBvdXRwLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgd2l0aCBvcGVuKG91dHAsICJ3IikgYXMgZjoKICAgICAgICAgICAganNvbi5kdW1wKHJlc3VsdHMsIGYsIGluZGVudD0yKQogICAgICAgIHByaW50KGYid3JvdGUge291dHB9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="
!nvidia-smi -L 2>/dev/null || echo "⚠️  set Runtime ▸ Change runtime type ▸ GPU (T4)"
import os, subprocess, base64
from getpass import getpass
REPO, BRANCH, DEST = "Dypatterson/Neuro-AI", "consolidation/role-structure", "/content/Neuro-AI"
if not os.path.isdir(DEST):
    tok = getpass("GitHub token (press Enter if the repo is public): ").strip()
    url = f"https://{tok}@github.com/{REPO}.git" if tok else f"https://github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, DEST], check=True)
else:
    subprocess.run(["git", "-C", DEST, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", f"origin/{BRANCH}"], check=True)
subprocess.run(["pip", "-q", "install", "datasets"], check=True)
# Self-contained: materialize the exp-57 head-to-head harness if it is not yet on the branch.
HARNESS = f"{DEST}/experiments/57_e2e_integration_wiring_check.py"
if not os.path.exists(HARNESS):
    open(HARNESS, "wb").write(base64.b64decode(EXP57_B64))
    print("ℹ️  materialized exp-57 (not yet pushed to the branch)")
else:
    print("✓ exp-57 present on branch")
print("HEAD:", subprocess.check_output(["git", "-C", DEST, "log", "-1", "--oneline"]).decode().strip())


## 2 · Run the head-to-head @ D=4096 WikiText-2
_A single **CUDA-isolated subprocess** (the parent notebook never touches CUDA). The `wikitext` preset = Report-055/056 config: D=4096, N=1000, max_vocab=2000, W=6, 3 seeds, obs 1/2/3, in-sample._

In [ ]:
import os, subprocess, time
os.makedirs("/content/results", exist_ok=True)
OUT = "/content/results/wiring_wikitext_D4096.json"
env = {**os.environ, "PYTHONPATH": "src"}
if not os.path.exists(OUT):
    t = time.time()
    cmd = ["python", "experiments/57_e2e_integration_wiring_check.py",
           "--corpora", "wikitext", "--device", "cuda", "--out", OUT]
    r = subprocess.run(cmd, cwd="/content/Neuro-AI", env=env, capture_output=True, text=True)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print("STDERR:\n", r.stderr[-3500:]); raise RuntimeError("wiring check failed")
    print(f"\n✓ {time.time()-t:.0f}s")
else:
    print("cached:", OUT)


## 3 · The wiring certificate — bit-identical A == B + role-Selectivity-Δ vs the 055/056 anchor

In [ ]:
import json, pandas as pd
R = json.load(open(OUT)); P = R["pooled"]
OBS = sorted({int(k.split("obs")[1]) for k in P})
rows = []
for o in OBS:
    c = P[f"wikitext_obs{o}"]; b = c["integrated_B"]; a = c["standalone_A"]
    rows.append(dict(
        obs=o,
        role_delta_B=round(b["delta"], 3),
        role_delta_A=round(a["delta"], 3),
        B_eq_A=c["B_equals_A"],
        true_rate_B=round(b["true_rate"], 3),
        deranged=round(b["shuffled_rate"], 3),
        two_floor=b["two_floor_pass"],
        anchor_056=c["report_role_delta"],
        A_minus_report=(round(c["A_minus_report"], 3) if c["A_minus_report"] is not None else None),
        random_cb=round(c["rand_rate"], 4),
        chance=round(b["chance"], 4),
        n=c["n_pooled"]))
df = pd.DataFrame(rows); display(df)
print("ALL basin indices bit-identical (A == B), every cell:", R["all_basin_indices_bit_identical"])


## 4 · The money plot — the integrated path lands exactly on the standalone reference 📈
_The standalone-A dashed line sits **under** the integrated-B line (they coincide — bit-identical), and both hit the Report-056 ⭐ anchors. The deranged + random-codebook arms collapse._

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
rdB = [r["role_delta_B"] for r in rows]; rdA = [r["role_delta_A"] for r in rows]
anc = [r["anchor_056"] for r in rows]; der = [r["deranged"] for r in rows]
rnd = [r["random_cb"] for r in rows]; tr = [r["true_rate_B"] for r in rows]
fig, ax = plt.subplots(figsize=(8.8, 5.4))
ax.plot(OBS, tr, "o-", color="#2a9d8f", lw=2.9, label="write+L2 true recall (integrated B) ≈ ceiling")
ax.plot(OBS, rdA, "--", color="#e9c46a", lw=4.0, alpha=.95, label="role-Δ (standalone A)")
ax.plot(OBS, rdB, "o-", color="#264653", lw=2.2, label="role-Δ (integrated B) — overlies A")
ax.scatter(OBS, anc, marker="*", s=260, color="#c1121f", zorder=6, label="Report-056 anchor")
ax.plot(OBS, der, "k:", alpha=.6, label="position-deranged floor")
ax.plot(OBS, rnd, "o:", color="#999", label="random-codebook control → chance")
ax.set_xlabel("context positions in cue  (sparse → rich)"); ax.set_ylabel("Recall@1 / role-Selectivity-Δ")
ax.set_title("Integration wiring · D=4096 · WikiText-2 · integrated path == standalone (bit-identical)")
ax.set_xticks(OBS); ax.set_ylim(-0.02, 1.04); ax.legend(loc="center right", fontsize=8.3); ax.grid(alpha=.3)
nbit = sum(1 for r in rows if r["B_eq_A"])
ax.text(0.02, 0.97, f"A == B bit-identical: {nbit}/{len(rows)} cells",
        transform=ax.transAxes, fontsize=12, fontweight="bold", va="top",
        bbox=dict(boxstyle="round", fc="#e9f5ec", ec="#2a9d8f"))
plt.tight_layout(); plt.savefig("/content/results/wiring_certificate.png", bbox_inches="tight"); plt.show()


## 5 · 🔌 The certificate

In [ ]:
ok = R["all_basin_indices_bit_identical"]
print("Integration wiring certificate — D=4096 WikiText-2 (in-sample memorization):\n")
print("ALL basin indices bit-identical (A == B):", ok, "\n")
for r in rows:
    am = f"{r['A_minus_report']:+.3f}" if r["A_minus_report"] is not None else " n/a "
    print(f"  obs={r['obs']}: integrated role-Δ {r['role_delta_B']:.3f}  ==  standalone {r['role_delta_A']:.3f}  "
          f"(B==A {r['B_eq_A']}) | true {r['true_rate_B']:.3f} | two_floor {r['two_floor']} | "
          f"056-anchor {r['anchor_056']} (A−rep {am}) | rand {r['random_cb']:.4f} / chance {r['chance']:.4f}")
print()
print("🔌 WIRING CONFIRMED at D=4096 — the integrated OnlineCodebookUpdater path reproduces the\n"
      "   graduated mechanism BIT-IDENTICALLY at the real substrate dimension ✅" if ok
      else "❌ NOT bit-identical — the fold-in altered the computation; investigate before trusting it.")


## 6 · Save to Drive (optional)

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import shutil, os, datetime
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
dest = f"/content/drive/MyDrive/neuro-ai/results/wiring_certificate_{stamp}"; os.makedirs(dest, exist_ok=True)
df.to_csv(f"{dest}/wiring_table.csv", index=False)
for f in os.listdir("/content/results"): shutil.copy(f"/content/results/{f}", dest)
print("Saved →", dest)


---
Paste me the **§5 certificate** (or the table + plot). If `A == B` is bit-identical across all cells at D=4096, the integration is confirmed **behaviorally equivalent to the graduated mechanism at the real substrate dimension** — completing the Report-058 wiring claim end-to-end on GPU.

_Harness: `experiments/57_e2e_integration_wiring_check.py --corpora wikitext --device cuda`. Mechanism: `phase4/{hetero_write,decorrelator}.py`; integration: `phase34/online_codebook.py` (`observe(cue=)`/`consolidate_hetero()`/`recall_hetero()`). The key is the RAW masked cue — H bypasses the scene-MHN+unbind that corrupts slot_query at sparse cues._